In [319]:
import pandas as pd
import numpy as np
import os
import geopandas as gpd
import json
import plotly.express as px
import plotly.graph_objects as go

In [322]:
msa = gpd.read_file(r'C:\Users\annas\OneDrive\Github\plotly_bls_map\cbsa_shape\cb_2025_us_cbsa_500k.shp')
msa = msa.to_crs(epsg=4326)
msa.to_file(r'C:\Users\annas\OneDrive\Github\plotly_bls_map\cbsa_shape\cb_2025_us_cbsa_500k.geojson', driver='GeoJSON', index=False)

msa = (r'C:\Users\annas\OneDrive\Github\plotly_bls_map\cbsa_shape\cb_2025_us_cbsa_500k.geojson')

with open(msa) as f:
    msa = json.load(f)

In [ ]:
states = pd.read_excel('state_abr.xlsx')
dir_path = r'C:\Users\annas\OneDrive\Github\plotly_bls_map\state_employment_data'
files = [f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f))]

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

#make a dictionary of each state name and DF
df_name = states['ABR']

datasets = {}
for i in range(0, len(df_name)):
    a = df_name[i]

    if f'{df_name[i]}.xlsx' in os.listdir(dir_path):
        datasets[a] = pd.read_excel(f'{dir_path}/{a}.xlsx', skiprows = 2)
    else:
        continue

def add_county_column(df):
    df['MSA'] = df['Area'].apply(lambda x: str(x)[:-4])

def industry_percentage_by_msa(df):
    df['Percent'] = (df['Employment (1)'] / df.groupby('MSA')['Employment (1)'].transform('sum') * 100)

def replace_non_reported(df):
    df['Employment (1)'] = df['Employment (1)'].replace('(8) -', None)

for k, v in datasets.items():
    replace_non_reported(v)
    add_county_column(v)
    industry_percentage_by_msa(v)

msa = msa.rename(columns = {'NAME':'Area'})

In [293]:
fig = px.choropleth_map(texas_mgmt, 
                            geojson=msa.__geo_interface__, 
                            locations='Area', 
                            color='Percent',
                           color_continuous_scale="sunsetdark",
                           range_color=(0, 12),
                           map_style="carto-positron",
                           zoom=3, center = {"lat": 37.0902, "lon": -95.7129},
                           opacity=1
                          )